In [ ]:
import torch
if '2.7' not in torch.__version__:
    import subprocess, sys
    subprocess.run([sys.executable,'-m','pip','install','-q',
        'torch==2.7.0','torchvision','torchaudio',
        '--index-url','https://download.pytorch.org/whl/cu128'],
        capture_output=True)
    import IPython; IPython.Application.instance().kernel.do_shutdown(restart=True)
else:
    print(f'PyTorch {torch.__version__} -- sm_120 ready.')


# GWNet v24 — PEMS08 | Target MAE < 13.114

**v23 had the right architecture. Two precise bugs killed it:**
1. `OneCycleLR max_lr=2e-3` — model destabilized at peak LR (ep15: MAE worsened from 15.6→16.2)
2. **No GRU** — MGRC module from the MD-GRTN paper adds GRU after GCN; dilated conv alone misses sequential dependencies

**v24 fixes (everything else from v23 kept exactly):**
- LR: AdamW(5e-4) + linear warmup (3 ep) + ReduceLROnPlateau(p=15, f=0.65, min=5e-6)
- GRU(d_model→d_gru=128) on residual stream x after WaveBlocks → added to skip_agg
- grad_clip 5.0→3.0, weight_decay 1e-3→1e-4


In [ ]:
import os, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import pandas as pd

SEED = 42
def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
    os.environ['PYTHONHASHSEED'] = str(seed)
    print(f'Seed: {seed} OK')

set_seed()
print('PyTorch:', torch.__version__)
print('CUDA   :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU    :', torch.cuda.get_device_name(0))
    print('VRAM   :', round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), 'GB')

In [ ]:
class Config:
    # ── Paths ──────────────────────────────────────────────────────
    data_path    = "/kaggle/input/datasets/piyush1718s/pems08/PEMS08.npz"
    adj_csv_path = "/kaggle/input/datasets/piyush1718s/pems08csv/PEMS08.csv"
    best_path    = 'gwnet_v24_best.pt'

    # ── Dataset ────────────────────────────────────────────────────
    num_nodes   = 170
    in_features = 3
    seq_len     = 16     # 80 min input
    pred_len    = 12
    feature_idx = 0
    train_ratio = 0.7
    val_ratio   = 0.1
    # 3-stream offsets (5-min steps)
    HOUR_OFFSET = 12     # 1h ago
    DAY_OFFSET  = 288    # 24h ago

    # ── Model ──────────────────────────────────────────────────────
    d_model    = 128     # FIX: was 96
    d_skip     = 256     # leaner skip (was 512 — wasteful)
    d_end      = 512
    d_time     = 64
    # 8 blocks, dilations give receptive field = (4-1)*(1+2+4+8+1+2+4+8)=90 steps
    DILATIONS  = [1, 2, 4, 8, 1, 2, 4, 8]
    kernel_size = 4      # wider temporal kernel
    adp_emb    = 20      # FIX: was 10
    gcn_order  = 3
    n_supports = 3
    dropout    = 0.10

    # ── Training — THE critical fix ────────────────────────────────
    batch_size   = 48
    lr           = 5e-4   # AdamW base lr (warmup + plateau)
    epochs       = 250    # FIX: was 55
    patience     = 60     # FIX: was 20
    weight_decay = 1e-4
    d_gru        = 128    # GRU hidden dim (MGRC)

cfg    = Config()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'GWNet v24 | d={cfg.d_model} d_skip={cfg.d_skip} d_gru={cfg.d_gru} layers={len(cfg.DILATIONS)}')
print(f'seq={cfg.seq_len} kernel={cfg.kernel_size} gcn_order={cfg.gcn_order}')
print(f'epochs={cfg.epochs} patience={cfg.patience} | {device}')

In [ ]:
def build_sparse_adj(csv_path, num_nodes):
    """
    Build sparse Gaussian-kernel adjacency.

    CRITICAL FIX vs v20-3-o:
    v20 did: A_raw[i,j] = c, then exp(-(A_raw**2)/sigma**2) on FULL matrix.
    For non-edge pairs: A_raw[i,j]=0 → exp(0)=1.0 (highest possible weight!)
    For real edges: exp(-(315/216)^2) = 0.12 (lower than non-edges!)
    Result: avg degree=169, weights INVERTED. GCN sees nothing meaningful.

    FIX: Only apply Gaussian where actual edge exists:
      w[i,j] = exp(-c^2/sigma^2)  if road edge exists
      w[i,j] = 0                  otherwise
    Result: avg degree=3.2, correct spatial structure.
    """
    A = np.zeros((num_nodes, num_nodes), dtype=np.float32)
    try:
        df = pd.read_csv(csv_path, header=None, skiprows=1)
        df.columns = ['from', 'to', 'cost']
        df[['from','to']] = df[['from','to']].astype(int)
        df['cost'] = df['cost'].astype(float)
        sigma = df['cost'].std()
        for _, r in df.iterrows():
            i, j, c = int(r['from']), int(r['to']), float(r['cost'])
            if i < num_nodes and j < num_nodes:
                # Only fill where edge EXISTS — zero entries stay zero
                w = float(np.exp(-(c**2) / (sigma**2 + 1e-8)))
                A[i,j] = w;  A[j,i] = w
        nnz = int((A > 0).sum())
        print(f'Sparse adj: edges={len(df)} nnz={nnz} avg_deg={nnz/num_nodes:.1f}')
    except Exception as e:
        print(f'Adj fallback to identity: {e}')
        np.fill_diagonal(A, 1.0)
    return A


def load_pems08(cfg):
    raw  = np.load(cfg.data_path)
    data = raw['data'].astype(np.float32)    # (T, N, 3)
    T, N, F = data.shape
    print(f'Data: {data.shape}')
    mean_np = data.mean(axis=0)
    std_np  = data.std(axis=0) + 1e-8
    data_norm = (data - mean_np) / std_np
    A_raw = build_sparse_adj(cfg.adj_csv_path, N)
    return data_norm, mean_np, std_np, A_raw


class TrafficDataset3S(Dataset):
    """
    3-stream dataset: recent | 1h-ago | 24h-ago
    Each stream gives seq_len steps of (N, 3) features.
    Requires i >= DAY_OFFSET for valid 24h lookback.
    """
    def __init__(self, data, seq_len, pred_len, feature_idx,
                 hour_offset, day_offset, split_start=0, split_end=None):
        self.data        = data
        self.seq_len     = seq_len
        self.pred_len    = pred_len
        self.feat_idx    = feature_idx
        self.hour_offset = hour_offset
        self.day_offset  = day_offset
        T = len(data)
        split_end = split_end or T
        # Enforce minimum start for day lookback
        eff_start = max(split_start, day_offset)
        self.indices = list(range(eff_start, split_end - seq_len - pred_len + 1))

    def __len__(self): return len(self.indices)

    def __getitem__(self, idx):
        i, S = self.indices[idx], self.seq_len
        x_rec  = self.data[i              : i+S             ].copy()
        x_hour = self.data[i-self.hour_offset : i-self.hour_offset+S].copy()
        x_day  = self.data[i-self.day_offset  : i-self.day_offset+S ].copy()
        y      = self.data[i+S : i+S+self.pred_len, :, self.feat_idx].copy()
        tod    = np.array([(i+t) % 288      for t in range(S)], dtype=np.int64)
        dow    = np.array([((i+t)//288) % 7 for t in range(S)], dtype=np.int64)
        return (torch.from_numpy(x_rec),  torch.from_numpy(x_hour),
                torch.from_numpy(x_day),  torch.from_numpy(y),
                torch.from_numpy(tod),    torch.from_numpy(dow))


def build_dataloaders(cfg):
    set_seed()
    data, mean_np, std_np, A_raw = load_pems08(cfg)
    T  = len(data)
    t1 = int(T * cfg.train_ratio)
    t2 = int(T * (cfg.train_ratio + cfg.val_ratio))
    kw    = dict(batch_size=cfg.batch_size, num_workers=2, pin_memory=True)
    ds_kw = dict(data=data, seq_len=cfg.seq_len, pred_len=cfg.pred_len,
                 feature_idx=cfg.feature_idx,
                 hour_offset=cfg.HOUR_OFFSET, day_offset=cfg.DAY_OFFSET)
    dl_tr = DataLoader(TrafficDataset3S(**ds_kw, split_start=0,  split_end=t1), shuffle=True,  **kw)
    dl_va = DataLoader(TrafficDataset3S(**ds_kw, split_start=t1, split_end=t2), shuffle=False, **kw)
    dl_te = DataLoader(TrafficDataset3S(**ds_kw, split_start=t2, split_end=T),  shuffle=False, **kw)
    print(f'Train={len(dl_tr.dataset)} Val={len(dl_va.dataset)} Test={len(dl_te.dataset)}')
    return dl_tr, dl_va, dl_te, mean_np, std_np, A_raw

print('Data + 3-stream dataset ready.')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# DIFFUSION GCN — K-hop spatial aggregation
# ══════════════════════════════════════════════════════════════════
class DiffusionGCN(nn.Module):
    def __init__(self, d_in, d_out, n_supports=3, order=2, dropout=0.1):
        super().__init__()
        self.mlp   = nn.Linear(d_in * (1 + n_supports * order), d_out)
        self.drop  = nn.Dropout(dropout)
        self.order = order

    def forward(self, x, supports):  # x: (B*S, N, d)
        hs = [x]
        for A in supports:
            h = x
            for _ in range(self.order):
                h = torch.einsum('nm,bmd->bnd', A, h)
                hs.append(h)
        return self.drop(self.mlp(torch.cat(hs, dim=-1)))


# ══════════════════════════════════════════════════════════════════
# WAVE BLOCK — gated TCN + GCN + skip/residual
# ══════════════════════════════════════════════════════════════════
class WaveBlock(nn.Module):
    def __init__(self, d_model, d_skip, kernel_size, dilation,
                 n_supports, gcn_order, dropout):
        super().__init__()
        self.dilation    = dilation
        self.kernel_size = kernel_size
        conv_kw = dict(kernel_size=(1, kernel_size), dilation=(1, dilation))
        self.filter_conv = nn.Conv2d(d_model, d_model, **conv_kw)
        self.gate_conv   = nn.Conv2d(d_model, d_model, **conv_kw)
        self.gcn         = DiffusionGCN(d_model, d_model, n_supports, gcn_order, dropout)
        self.bn          = nn.BatchNorm2d(d_model)
        self.drop        = nn.Dropout(dropout)
        self.skip_conv   = nn.Conv2d(d_model, d_skip,  (1,1))
        self.res_conv    = nn.Conv2d(d_model, d_model, (1,1))

    def forward(self, x, supports):  # x: (B, d, N, S)
        residual = x
        pad   = (self.kernel_size - 1) * self.dilation
        x_pad = F.pad(x, [pad, 0])
        f = torch.tanh   (self.filter_conv(x_pad))
        g = torch.sigmoid(self.gate_conv  (x_pad))
        x = self.drop(f * g)
        B, d, N, S = x.shape
        xg = x.permute(0,3,2,1).reshape(B*S, N, d)
        xg = self.gcn(xg, supports)
        x  = xg.reshape(B,S,N,d).permute(0,3,2,1)
        x  = self.bn(x)
        skip = self.skip_conv(x)
        x    = self.res_conv(x) + residual
        return x, skip


# ══════════════════════════════════════════════════════════════════
# PERIOD FUSION — fuse hourly+daily into recent stream
# Cross-attention: recent queries, hourly+daily keys/values
# ══════════════════════════════════════════════════════════════════
class PeriodFusion(nn.Module):
    def __init__(self, d_model, n_heads=4, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_head = d_model // n_heads
        self.n_heads = n_heads
        self.q  = nn.Linear(d_model, d_model)
        self.k  = nn.Linear(d_model, d_model)
        self.v  = nn.Linear(d_model, d_model)
        self.o  = nn.Linear(d_model, d_model)
        self.norm = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)

    def forward(self, h_rec, h_ctx):  # each: (B, d, N, S)
        B, d, N, S = h_rec.shape
        nh, dh = self.n_heads, self.d_head

        def flat(t):  # (B,d,N,S) → (B*N, S, d)
            return t.permute(0,2,3,1).reshape(B*N, S, d)

        q = flat(h_rec);  kv = flat(h_ctx)
        Q = self.q(q).reshape(B*N, S, nh, dh).transpose(1,2)
        K = self.k(kv).reshape(B*N, S, nh, dh).transpose(1,2)
        V = self.v(kv).reshape(B*N, S, nh, dh).transpose(1,2)
        a = F.softmax(Q @ K.transpose(-2,-1) / dh**0.5, dim=-1)
        o = (self.drop(a) @ V).transpose(1,2).reshape(B*N, S, d)
        o = self.norm(q + self.o(o))                  # (B*N, S, d)
        return o.reshape(B, N, S, d).permute(0,3,1,2) # (B, d, N, S)  ← FIX: permute(0,3,1,2)


# ══════════════════════════════════════════════════════════════════
# GWNET v23
#
# Key fixes vs v20-3-o:
#   1. Sparse adjacency (avg_deg 169→3.2): GCN finally sees real roads
#   2. Single OneCycleLR (no cosine restart at ep50 that killed v20)
#   3. 3-stream input: recent + hourly (1h ago) + daily (24h ago)
#   4. d_model: 96→128, kernel_size: 2→4, adp_emb: 10→20
#   5. Learnable skip weights instead of uniform sum
# ══════════════════════════════════════════════════════════════════
class GWNetV24(nn.Module):
    def __init__(self, cfg, A_raw):
        super().__init__()
        N, d = cfg.num_nodes, cfg.d_model

        # ── Sparse adjacency (FIXED) ──────────────────────────────
        A = torch.FloatTensor(A_raw)
        D_f = A.sum(1, keepdim=True).clamp(min=1e-8)
        D_b = A.T.sum(1, keepdim=True).clamp(min=1e-8)
        self.register_buffer('A_fwd', A   / D_f)
        self.register_buffer('A_bwd', A.T / D_b)

        # ── Adaptive adjacency (learnable) ────────────────────────
        self.E1 = nn.Parameter(torch.randn(N, cfg.adp_emb) * 0.01)
        self.E2 = nn.Parameter(torch.randn(N, cfg.adp_emb) * 0.01)

        # ── Input projections — one per stream ────────────────────
        self.sc_rec  = nn.Conv2d(cfg.in_features, d, (1,1))
        self.sc_hour = nn.Conv2d(cfg.in_features, d, (1,1))
        self.sc_day  = nn.Conv2d(cfg.in_features, d, (1,1))
        self.node_emb = nn.Parameter(torch.randn(1, d, N, 1) * 0.01)

        # ── Temporal embeddings ───────────────────────────────────
        self.tod_emb   = nn.Embedding(288, cfg.d_time)
        self.dow_emb   = nn.Embedding(7,   cfg.d_time)
        self.time_proj = nn.Linear(cfg.d_time * 2, d)

        # ── Period fusion: fuse hourly+daily into recent ──────────
        self.fuse_hour = PeriodFusion(d, n_heads=4, dropout=cfg.dropout)
        self.fuse_day  = PeriodFusion(d, n_heads=4, dropout=cfg.dropout)
        self.fuse_proj = nn.Sequential(
            nn.Conv2d(d * 3, d, (1,1)), nn.ReLU()
        )

        # ── WaveBlocks ────────────────────────────────────────────
        self.blocks = nn.ModuleList([
            WaveBlock(d, cfg.d_skip, cfg.kernel_size, dil,
                      cfg.n_supports, cfg.gcn_order, cfg.dropout)
            for dil in cfg.DILATIONS
        ])
        n_blocks = len(cfg.DILATIONS)

        # ── Learnable skip weights ────────────────────────────────
        # Each block's contribution is weighted by learned scalars
        self.skip_w = nn.Parameter(torch.ones(n_blocks))

        # ── GRU — sequential temporal modeling (MGRC in paper) ───
        # Runs on residual stream x after all WaveBlocks.
        # Captures long-range sequential dependencies that dilated
        # conv misses. Final hidden state → projected to skip space.
        self.d_gru    = cfg.d_gru
        self.gru      = nn.GRU(cfg.d_model, cfg.d_gru,
                               num_layers=1, batch_first=True)
        self.gru_proj = nn.Linear(cfg.d_gru, cfg.d_skip)

        # ── Output ───────────────────────────────────────────────
        self.end_conv1 = nn.Conv2d(cfg.d_skip, cfg.d_end,    (1,1))
        self.end_conv2 = nn.Conv2d(cfg.d_end,  cfg.pred_len, (1,1))

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.Linear)):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None: nn.init.zeros_(m.bias)
        nn.init.normal_(self.tod_emb.weight, std=0.01)
        nn.init.normal_(self.dow_emb.weight, std=0.01)

    def get_supports(self):
        A_adp = F.softmax(F.relu(self.E1 @ self.E2.T), dim=-1)
        return [self.A_fwd, self.A_bwd, A_adp]

    def forward(self, x_rec, x_hour, x_day, tod=None, dow=None):
        # x_*: (B, S, N, 3)
        def to4d(x): return x.permute(0,3,2,1)  # (B,3,N,S)

        h_rec  = self.sc_rec (to4d(x_rec )) + self.node_emb  # (B,d,N,S)
        h_hour = self.sc_hour(to4d(x_hour))
        h_day  = self.sc_day (to4d(x_day ))

        # Temporal embedding → inject into recent stream
        if tod is not None:
            te = torch.cat([self.tod_emb(tod), self.dow_emb(dow)], dim=-1)  # (B,S,2*d_t)
            te = self.time_proj(te).permute(0,2,1).unsqueeze(2)              # (B,d,1,S)
            h_rec = h_rec + te

        # Period fusion: attend to hourly and daily context
        h_h = self.fuse_hour(h_rec, h_hour)  # (B,d,N,S)
        h_d = self.fuse_day (h_rec, h_day )
        # Concat and project back to d
        x = self.fuse_proj(torch.cat([h_rec, h_h, h_d], dim=1))  # (B,d,N,S)

        # WaveBlocks with learnable skip weights
        supports = self.get_supports()
        skips    = []
        for blk in self.blocks:
            x, skip = blk(x, supports)
            skips.append(skip)

        # Weighted skip aggregation (last timestep)
        w = F.softmax(self.skip_w, dim=0)                         # (n_blocks,)
        skip_agg = sum(w[i] * skips[i][..., -4:].mean(-1, keepdim=True)
                       for i in range(len(self.blocks)))           # (B,d_skip,N,1)

        # GRU on residual stream x — sequential temporal context
        # x: (B, d_model, N, S) after all WaveBlocks
        B2, d2, N2, S2 = x.shape
        x_gru = x.permute(0,2,3,1).reshape(B2*N2, S2, d2)        # (B*N, S, d)
        _, h_n = self.gru(x_gru)                                  # (1, B*N, d_gru)
        h_n = h_n.squeeze(0).reshape(B2, N2, self.d_gru)         # (B, N, d_gru)
        gru_skip = self.gru_proj(h_n)                             # (B, N, d_skip)
        gru_skip = gru_skip.permute(0,2,1).unsqueeze(-1)          # (B,d_skip,N,1)
        skip_agg = skip_agg + gru_skip                            # residual add

        out = F.relu(self.end_conv1(F.relu(skip_agg)))            # (B,d_end,N,1)
        out = self.end_conv2(out)                                  # (B,pred_len,N,1)
        return out.squeeze(-1)                                     # (B,pred_len,N)


print('GWNetV24 defined (+ GRU branch).')

In [ ]:
def masked_mae(pred, true):
    mask = (true.abs() > 0.0).float()
    return (torch.abs(pred - true) * mask).sum() / (mask.sum() + 1e-8)

def masked_rmse(pred, true):
    mask = (true.abs() > 0.0).float()
    return (((pred-true)**2 * mask).sum() / (mask.sum() + 1e-8)).sqrt()

def masked_mape(pred, true, thresh=10.0):
    mask = (true.abs() > thresh).float()
    if mask.sum() < 1: return torch.tensor(0.0, device=pred.device)
    return (torch.abs((pred-true)/(true.abs()+1e-8)) * mask).sum() / mask.sum() * 100

def huber_loss(pred, true, delta=1.0):
    err  = torch.abs(pred - true)
    loss = torch.where(err < delta, 0.5*err**2, delta*(err - 0.5*delta))
    return loss.mean()

print('Metrics ready.')

In [ ]:
set_seed()
dl_train, dl_val, dl_test, mean_np, std_np, A_raw = build_dataloaders(cfg)

mean_flow = torch.from_numpy(mean_np[:, cfg.feature_idx]).to(device)
std_flow  = torch.from_numpy(std_np [:, cfg.feature_idx]).to(device)

# Build model (adjacency passed at construction)
model = GWNetV23(cfg, A_raw).to(device)
params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'GWNet v23 | {params:,} parameters')

# Smoke test
with torch.no_grad():
    B = 2
    dummy = lambda: torch.randn(B, cfg.seq_len, cfg.num_nodes, cfg.in_features).to(device)
    td = torch.randint(0, 288, (B, cfg.seq_len)).to(device)
    dw = torch.randint(0, 7,   (B, cfg.seq_len)).to(device)
    out = model(dummy(), dummy(), dummy(), td, dw)
    ok = list(out.shape) == [B, cfg.pred_len, cfg.num_nodes]
    print(f'Output: {out.shape}  {chr(10003) if ok else "FAIL"}')
    print(f'GPU mem: {torch.cuda.memory_allocated()/1e9:.2f} GB')
    torch.cuda.empty_cache()

In [ ]:
scaler = torch.amp.GradScaler('cuda')

def train_epoch(model, loader, optimizer, device):
    model.train()
    total = 0.0
    for x_rec, x_hour, x_day, y, tod, dow in loader:
        x_rec  = x_rec.to(device,  non_blocking=True)
        x_hour = x_hour.to(device, non_blocking=True)
        x_day  = x_day.to(device,  non_blocking=True)
        y      = y.to(device,      non_blocking=True)
        tod    = tod.to(device,    non_blocking=True)
        dow    = dow.to(device,    non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda'):
            pred = model(x_rec, x_hour, x_day, tod, dow)
            loss = huber_loss(pred, y)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 3.0)
        scaler.step(optimizer)
        scaler.update()
        total += loss.item()
    return total / len(loader)


@torch.no_grad()
def eval_epoch(model, loader, device, mean_flow, std_flow):
    model.eval()
    maes, rmses, mapes = [], [], []
    for x_rec, x_hour, x_day, y, tod, dow in loader:
        x_rec  = x_rec.to(device,  non_blocking=True)
        x_hour = x_hour.to(device, non_blocking=True)
        x_day  = x_day.to(device,  non_blocking=True)
        y      = y.to(device,      non_blocking=True)
        tod    = tod.to(device,    non_blocking=True)
        dow    = dow.to(device,    non_blocking=True)
        with torch.amp.autocast('cuda'):
            pred = model(x_rec, x_hour, x_day, tod, dow)
        pd_ = pred.float() * std_flow[None,None,:] + mean_flow[None,None,:]
        yd_ = y.float()    * std_flow[None,None,:] + mean_flow[None,None,:]
        maes.append (masked_mae (pd_, yd_).item())
        rmses.append(masked_rmse(pd_, yd_).item())
        mapes.append(masked_mape(pd_, yd_).item())
    return np.mean(maes), np.mean(rmses), np.mean(mapes)

print('Train / eval ready.')

In [ ]:
set_seed()
model = GWNetV24(cfg, A_raw).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

# ════════════════════════════════════════════════════════════
# LR SCHEDULE FIX — the only reason v23 failed
#
# OneCycleLR(max_lr=2e-3) peaked at ep20 with lr=2e-3:
#   ep10: MAE=15.6 (good), ep15: MAE=16.2 (WORSE — LR too high!)
#
# v24: linear warmup (3ep) then ReduceLROnPlateau.
# Patience=15 gives model time to escape local minima.
# min_lr=5e-6 allows very fine final tuning.
# ════════════════════════════════════════════════════════════
def lr_warmup(epoch):
    return min(1.0, (epoch + 1) / 3)

warmup_sched  = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_warmup)
plateau_sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=15, factor=0.65, min_lr=5e-6, verbose=False)

best_val_mae  = float('inf')
best_val_rmse = float('inf')
best_val_mape = float('inf')
patience_cnt  = 0
history       = {'train_loss':[], 'val_mae':[], 'val_rmse':[], 'val_mape':[]}

params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'GWNet v24 | {params:,} params')
print(f'd={cfg.d_model} d_skip={cfg.d_skip} d_gru={cfg.d_gru} | dilations={cfg.DILATIONS}')
print(f'kernel={cfg.kernel_size} gcn_order={cfg.gcn_order} adp_emb={cfg.adp_emb}')
print(f'3-stream + GRU | AdamW lr={cfg.lr} + plateau(p=15,f=0.65)')
print(f'epochs={cfg.epochs} patience={cfg.patience}')
print(f'Baseline -> MAE=13.114 | RMSE=22.623 | MAPE=8.471%')
print('='*70)

for epoch in range(1, cfg.epochs + 1):
    train_loss = train_epoch(model, dl_train, optimizer, device)
    val_mae, val_rmse, val_mape = eval_epoch(
        model, dl_val, device, mean_flow, std_flow)

    # LR step: warmup first 3ep, then plateau
    if epoch <= 3:
        warmup_sched.step()
    else:
        plateau_sched.step(val_mae)

    history['train_loss'].append(train_loss)
    history['val_mae'].append(val_mae)
    history['val_rmse'].append(val_rmse)
    history['val_mape'].append(val_mape)

    tag = ''
    if val_mae < best_val_mae:
        best_val_mae  = val_mae
        best_val_rmse = val_rmse
        best_val_mape = val_mape
        patience_cnt  = 0
        torch.save(model.state_dict(), cfg.best_path)
        tag = '  <- best'
    else:
        patience_cnt += 1
        if patience_cnt >= cfg.patience:
            print(f'Early stop at epoch {epoch}.')
            break

    lr_now = optimizer.param_groups[0]['lr']
    beat   = ' *** BEAT BASELINE! ***' if val_mae < 13.114 else ''
    if epoch % 5 == 0 or tag:
        print(f'Ep {epoch:03d} | Loss={train_loss:.4f} | '
              f'MAE={val_mae:.3f} RMSE={val_rmse:.3f} MAPE={val_mape:.2f}% '
              f'lr={lr_now:.1e}{tag}{beat}')

print(f'\nBest Val: MAE={best_val_mae:.3f} RMSE={best_val_rmse:.3f} MAPE={best_val_mape:.2f}%')
print(f'Baseline: MAE=13.114   RMSE=22.623   MAPE=8.471%')


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(history['train_loss'], color='steelblue')
axes[0].set_title('Train Loss (Huber)'); axes[0].set_xlabel('Epoch')
axes[1].plot(history['val_mae'], color='tab:orange', label='Val MAE')
axes[1].axhline(14.438, color='orange', ls=':', label='v20 val=14.438')
axes[1].axhline(13.114, color='red',    ls='--', label='Baseline 13.114')
axes[1].set_title('Validation MAE'); axes[1].set_xlabel('Epoch'); axes[1].legend()
axes[2].plot(history['val_rmse'], color='tab:green', label='Val RMSE')
axes[2].axhline(22.623, color='red', ls='--', label='Baseline 22.623')
axes[2].set_title('Validation RMSE'); axes[2].set_xlabel('Epoch'); axes[2].legend()
plt.tight_layout(); plt.savefig('training_curves_v23.png', dpi=150); plt.show()

w = F.softmax(model.skip_w, dim=0).detach().cpu().numpy()
print('\nLearned skip weights per block:')
for i, (wi, d) in enumerate(zip(w, cfg.DILATIONS)):
    print(f'  Block {i+1} dil={d:2d}: {wi:.4f}  {"|"+"█"*int(wi*120)}')

In [ ]:
model.load_state_dict(torch.load(cfg.best_path, map_location=device))

@torch.no_grad()
def paper_eval(model, loader, device, mean_flow, std_flow):
    model.eval()
    all_pred, all_true = [], []
    for x_rec, x_hour, x_day, y, tod, dow in loader:
        x_rec  = x_rec.to(device);   x_hour = x_hour.to(device)
        x_day  = x_day.to(device);   y      = y.to(device)
        tod    = tod.to(device);      dow    = dow.to(device)
        with torch.amp.autocast('cuda'):
            pred = model(x_rec, x_hour, x_day, tod, dow)
        pd_ = pred.float() * std_flow[None,None,:] + mean_flow[None,None,:]
        yd_ = y.float()    * std_flow[None,None,:] + mean_flow[None,None,:]
        all_pred.append(pd_.cpu()); all_true.append(yd_.cpu())

    P = torch.cat(all_pred); Y = torch.cat(all_true)
    mae  = torch.abs(P-Y).mean().item()
    rmse = ((P-Y)**2).mean().sqrt().item()
    mask = Y.abs() > 10.0
    mape = (torch.abs((P[mask]-Y[mask])/(Y[mask].abs()+1e-8))).mean().item()*100

    print('\n' + '='*62)
    print('  TEST RESULTS — GWNet v23 — all 12 steps')
    print('='*62)
    print(f'  MAE  : {mae:.3f}   v20: 14.438  baseline: 13.114  d={mae-13.114:+.3f}')
    print(f'  RMSE : {rmse:.3f}   v20: ~24.4   baseline: 22.623  d={rmse-22.623:+.3f}')
    print(f'  MAPE : {mape:.3f}%  v20: ~8.5%   baseline:  8.471%  d={mape-8.471:+.3f}%')
    all_beat = (mae < 13.114) and (rmse < 22.623) and (mape < 8.471)
    print(f'\n  All baselines beaten: {"YES" if all_beat else "partial"}')
    print('='*62)
    return mae, rmse, mape

paper_eval(model, dl_test, device, mean_flow, std_flow)

In [ ]:
@torch.no_grad()
def horizon_eval(model, loader, device, mean_flow, std_flow):
    model.eval()
    buf = {h:{'mae':[],'rmse':[],'mape':[]} for h in [2,5,11]}
    for x_rec, x_hour, x_day, y, tod, dow in loader:
        x_rec  = x_rec.to(device);   x_hour = x_hour.to(device)
        x_day  = x_day.to(device);   y      = y.to(device)
        tod    = tod.to(device);      dow    = dow.to(device)
        with torch.amp.autocast('cuda'):
            pred = model(x_rec, x_hour, x_day, tod, dow)
        pd_ = pred.float()*std_flow[None,None,:]+mean_flow[None,None,:]
        yd_ = y.float()   *std_flow[None,None,:]+mean_flow[None,None,:]
        for h in buf:
            buf[h]['mae'].append (masked_mae (pd_[:,h,:], yd_[:,h,:]).item())
            buf[h]['rmse'].append(masked_rmse(pd_[:,h,:], yd_[:,h,:]).item())
            buf[h]['mape'].append(masked_mape(pd_[:,h,:], yd_[:,h,:]).item())
    print(f"\n{'Horizon':>14} | {'MAE':>8} | {'RMSE':>8} | {'MAPE':>9}")
    print('-'*52)
    for h, lbl in zip([2,5,11], ['3-step(15m)','6-step(30m)','12-step(60m)']):
        m = {k:np.mean(v) for k,v in buf[h].items()}
        print(f"{lbl:>14} | {m['mae']:>8.3f} | {m['rmse']:>8.3f} | {m['mape']:>8.2f}%")

horizon_eval(model, dl_test, device, mean_flow, std_flow)